In [28]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTEN

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, accuracy_score, f1_score

In [2]:
data = pd.read_excel("../data/job_classification.ods", engine = "odf", dtype = str)

data.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [3]:
data = data.dropna(axis = 0)
data.shape

(8073, 6)

In [4]:
def filler_location(location):
    result = re.findall("\\,\\s[A-Z]{2}$", location)
    if len(result) > 0:
        return result[0][2:]
    else:
        return location


data["location"] = data["location"].apply(filler_location)

In [6]:
target = "career_level"
x = data.drop(target, axis = 1)
y = data[target]

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [8]:
transformers = ColumnTransformer(transformers = [
    ("title", TfidfVectorizer(stop_words="english"), "title"),
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("description", TfidfVectorizer(stop_words="english", ngram_range=(1,2)), "description"),
    ("function", OneHotEncoder(handle_unknown="ignore"), ["function"]),
    ("industry", TfidfVectorizer(stop_words="english"), "industry")
])

In [9]:
normal_model = Pipeline(steps=[
    ("transformer", transformers),
    ("classifier", LogisticRegression(max_iter = 1000))
])

In [11]:
normal_model.fit(x_train, y_train)
y_normal_predict = normal_model.predict(x_test)
print(classification_report(y_test, y_normal_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.51      0.40      0.45       192
         director_business_unit_leader       0.80      0.29      0.42        14
                   manager_team_leader       0.66      0.65      0.66       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.75      1615
                             macro avg       0.47      0.37      0.40      1615
                          weighted avg       0.74      0.75      0.74      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [26]:
balanced_model = Pipeline(steps=[
    ("transformer", transformers),
    ("classifier", LogisticRegression(max_iter = 1000, class_weight = "balanced")),
])

In [27]:
balanced_model.fit(x_train, y_train)
y_balanced_predict = normal_model.predict(x_test)
print(classification_report(y_test, y_balanced_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.52      0.41      0.46       192
         director_business_unit_leader       0.80      0.29      0.42        14
                   manager_team_leader       0.66      0.65      0.66       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.75      1615
                             macro avg       0.47      0.38      0.40      1615
                          weighted avg       0.74      0.75      0.75      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [22]:
over_sampling = SMOTEN(random_state=42, k_neighbors = 2, sampling_strategy = {
    "managing_director_small_medium_company" : 100,
    "specialist" : 100,
    "director_business_unit_leader" : 100,
    "bereichsleiter" : 1000
})
x_train, y_train = over_sampling.fit_resample(x_train, y_train)
print(y_train.value_counts())

career_level
senior_specialist_or_project_manager      3469
manager_team_leader                       2138
bereichsleiter                            1000
specialist                                 100
director_business_unit_leader              100
managing_director_small_medium_company     100
Name: count, dtype: int64


In [23]:
oversampling_model = Pipeline(
    steps=[
        ("transformers", transformers),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [24]:
oversampling_model.fit(x_train, y_train)

y_over_predict = oversampling_model.predict(x_test)

print(classification_report(y_test, y_over_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.51      0.45      0.48       192
         director_business_unit_leader       0.86      0.43      0.57        14
                   manager_team_leader       0.67      0.63      0.65       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.75      1615
                             macro avg       0.48      0.40      0.43      1615
                          weighted avg       0.74      0.75      0.75      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [25]:
results = pd.DataFrame({
    "Experiment": [
        "Normal",
        "Class Weight Balanced",
        "Random Oversampling"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_normal_predict),
        accuracy_score(y_test, y_balanced_predict),
        accuracy_score(y_test, y_over_predict)
    ],
    "Macro F1": [
        f1_score(y_test, y_normal_predict, average="macro"),
        f1_score(y_test, y_balanced_predict, average="macro"),
        f1_score(y_test, y_over_predict, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_test, y_normal_predict, average="weighted"),
        f1_score(y_test, y_balanced_predict, average="weighted"),
        f1_score(y_test, y_over_predict, average="weighted")
    ]
})

results

,Experiment,Accuracy,Macro F1,Weighted F1
0,Normal,0.75356,0.400277,0.744582
1,Class Weight Balanced,0.75356,0.400277,0.744582
2,Random Oversampling,0.75356,0.428502,0.746343


## Conclusion

SMOTEN achieved the highest Macro F1-score (0.429),
compared with 0.400 for the original Logistic Regression model and
0.400 for the class-weighted model.

The accuracy remained approximately unchanged at 75.4%, suggesting that
oversampling improved performance on minority classes without
substantially affecting overall classification accuracy.

However, extremely rare classes such as `specialist` and
`managing_director_small_medium_company` remained difficult to predict
because of their very small number of samples.

Therefore, SMOTEN was selected as the best imbalance
handling method for the current experiments.